Authors: Dr. Víctor Uc Cetina

Notebook requirements

Input file: **KB-01.txt**

For creating a KB.txt file, please use notebook: 

**41 Create KB.ipynb**

## Transformers

In [1]:
!pip install transformers

Defaulting to user installation because normal site-packages is not writeable


In [2]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
the_model = 'mrm8488/distill-bert-base-spanish-wwm-cased-finetuned-spa-squad2-es'
TOKENIZER = AutoTokenizer.from_pretrained(the_model, do_lower_case=False)
MODEL = AutoModelForQuestionAnswering.from_pretrained(the_model)

/Users/victoruccetina/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/victoruccetina/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of the model checkpoint at mrm8488/distill-bert-base-spanish-wwm-cased-finetuned-spa-squad2-es were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreT

## FastText

In [3]:
# Fasttext installation
!python3 --version
%pwd

# In a notebook, use %cd (plain "cd" is invalid Python).
# Do not use sudo; that is what asked for a password.
# The repo is already cloned under LLMs/Notebooks/fastText
%cd /Users/victoruccetina/Documents/code/inteligencia-artificial/LLMs/Notebooks/fastText
!python3 -m pip install .
%cd /Users/victoruccetina/Documents/code/inteligencia-artificial/LLMs/Notebooks

%pwd

Python 3.9.6
/Users/victoruccetina/Documents/code/inteligencia-artificial/LLMs/Notebooks/fastText


/Users/victoruccetina/Library/Python/3.9/lib/python/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


Defaulting to user installation because normal site-packages is not writeable
Processing ./.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for fasttext: filename=fasttext-0.9.2-cp39-cp39-macosx_26_0_universal2.whl size=673448 sha256=4a4ff4c4c382a1c07eeb5d07d789dd1ae9f93833775750c561f53cf9584619c1
  Stored in directory: /private/var/folders/q8/1mz_tyzs0l97fh18rhtfz2v40000gn/T/pip-ephem-wheel-cache-g5qfxdqt/wheels/a4/7d/1a/690117b60f178a41382bc807eda0677daffa27b3055e49e965
Successfully built fasttext
  Attempting uninstall: fasttext
    Found existing installation: fasttext 0.9.2
    Uninstalling fasttext-0.9.2:
      Successfully uninstalled fasttext-0.9.2
/Users/victoruccetina/Documents/code/inteligencia-artificial/LLMs/Notebooks


'/Users/victoruccetina/Documents/code/inteligencia-artificial/LLMs/Notebooks'

In [4]:
import fasttext 
import fasttext.util

#from google.colab import drive
#drive.mount('/content/drive')

model_file = 'cc.es.300.bin'
ft = fasttext.load_model(model_file)
ft.get_dimension()

300

In [5]:
import math
from scipy.spatial import distance as cos_distance

def sorting_key(e):
  return e['distance']

def distance(vec1, vec2):
    dist = 0
    for i in range(len(vec1)):
        dist += (vec1[i]-vec2[i]) ** 2
    dist = math.sqrt( dist )
    return dist

def search_fasttext(k, v1, entries, entries_vec, dist_metric):
    ## it calculates the k closest vectors to v1
    dist_list = list()
    idx_success = list()
    for i in range(len(entries)):
        e2 = entries[i]
        v2 = entries_vec[i]
        if dist_metric == "Euclidean":
          tmpDist = distance(v1,v2)
        elif dist_metric == "Cosine":
          tmpDist = cos_distance.cosine(v1,v2)
        dist_list.append({'idx': i, 'entry': e2, 'distance': tmpDist})
    dist_list.sort(key=sorting_key)
    for i in range(k):
        idx_success.append({'idx': dist_list[i].get('idx'), 'distance': dist_list[i].get('distance')})
    return idx_success

## Reading **KB** file

In [6]:
entries = list()
f = open("KB-01.txt", "r")
for one_line in f:
  #print(one_line.strip())
  entries.append(one_line.strip())

entries_vec = list()
for e in entries:
  entries_vec.append(ft.get_sentence_vector(e))

## Answering questions with the help of FastText and BERT!

In [13]:
#pregunta = "¿Cuántas capas ocultas tiene una red neuronal?"
#pregunta = "Cuantas neuronas tenemos?"
pregunta = "qué es una red neuronal artificial?"

In [14]:
query = pregunta
k = 20
#distance_metric = "Cosine"
distance_metric = "Euclidean"
query_vec = ft.get_sentence_vector(query)

idx_success = search_fasttext(k, query_vec, entries, entries_vec, distance_metric)

top_results = list()
for i in range(len(idx_success)):
  top_results.append( [ idx_success[i]['idx'], entries[ idx_success[i]['idx'] ], idx_success[i]['distance'] ] )

contexto = ""
for idx in range(len(top_results)):
  contexto += top_results[idx][1] + " "
print("Contexto:\n", contexto)

Contexto:
 Por otra parte, una red autoasociativa es una red cuya principal misión es reconstruir una determinada información de entrada que se presente incompleta o distorsionada (le asocia el dato almacenado más parecido) Luego debe decidirse si una red neuronal es adecuada para resolver dicho problema Por lo tanto la pregunta es ¿cómo puede entonces una red neuronal calcular una salida? La respuesta es sencilla; los datos tienen que ser codificados, o sea, deben hallarse valores apropiados para representar las características simbólicas (alto, bajo, adecuado, etc.) Esta es la red neuronal más antigua; utilizándose hoy en día para aplicación como identificador de patrones Cada grupo, que representaba una habilidad especial, fue conectado exactamente a una neurona en la primer capa oculta Una red neuronal es “un nuevo sistema para el tratamiento de la información, cuya unidad básica de procesamiento está inspirada en la célula fundamental del sistema nervioso humano: la neurona” Una r

In [15]:
nlp = pipeline('question-answering', model=MODEL, tokenizer=TOKENIZER)
salida = nlp(question=pregunta, context=contexto)
print(salida)
print(salida['answer'])

Device set to use mps:0


{'score': 0.08845935761928558, 'start': 2705, 'end': 2766, 'answer': 'tratar de predecir una correcta clasificación de los clientes'}
tratar de predecir una correcta clasificación de los clientes
